# EF · 微调的价值 + HuggingFace vs unsloth 框架对比（Qwen3-0.6B）

> **罗氏 · 大模型微调课程实操**｜任务：临床短句 → 结构化 JSON 抽取
> 上层设计见 `../docs/2026-08-14-finetuning-course-design.md`

## 这个 notebook 回答两个问题
**① 微调到底值不值？**（头条）在**同一留出集**上比 3 种方案：
- **不微调·零样本**：直接给 base Qwen3-0.6B 指令
- **不微调·少样本(3例)**：prompt 里塞 3 个示例
- **微调后**：LoRA SFT 过的模型

关键看点：指令给了字段名，却**没说 `sex` 要归一成 `M/F`**（原文是 "female"）。base 只能猜 → 精确匹配大量失败；微调把这个隐含约定「内化」进权重 → 干净稳定。这正是 F1 的命题：**行为/格式用微调，而且能免掉长 prompt（降本降延迟）**。

**② 微调框架 HF vs unsloth 差多少？**（次要）同数据、同 LoRA 配置、同步数下比 **训练时间 / 峰值显存 / 收敛 / 上手成本**。

## 执行方式
**全部内联在 cell 里执行**（无 `%%writefile`、无子进程）：数据/配置/结果都是内存变量，cell 间共享。每个阶段用一个函数封装，返回后局部的模型变量自动回收，再 `free()` 清显存。

- **目标环境**：AWS SageMaker Notebook，单卡 **T4（16G 显存）**；**fp16**（不支持 bf16）、注意力 **sdpa**（不支持 FA2）
- 仅用开源框架，**不依赖 SageMaker SDK**，可在任意带 GPU（≥8G）环境运行
- **从上往下顺序执行**（或 Restart & Run All）。改了哪个 cell 就重跑那个 cell 即可——不再有「改 notebook 不改磁盘脚本」的坑
- ⚠️ **务必按顺序：HF（第 7 节）必须在 unsloth（第 8 节）之前跑**。unsloth 一 import 就会 monkey-patch `trl.SFTTrainer`，之后 HF 路线就不再是原生 TRL 了

## 1. 安装依赖（首次运行一次）
**内核提示**：优先选带 GPU 的 PyTorch 内核（如 SageMaker `conda_pytorch_p310`），它已自带 torch+CUDA，可注释掉下方 `torch` 那行。若用 `conda_python3` 这类不带 torch 的内核，保留 `torch` 安装。
⚠️ 装完 `unsloth` 后**重启 kernel** 再继续——unsloth 会调整 transformers/trl 版本。

In [ ]:
# 首次运行执行一次；装完 unsloth 后建议【重启 kernel】再往下跑（unsloth 会调整依赖版本）
# 若当前内核已自带 torch（如 SageMaker 的 conda_pytorch_p310 内核），可注释掉下一行
%pip install -q torch
%pip install -q -U transformers trl peft datasets accelerate
# unsloth：Colab 免费版即 T4，官方支持 T4。受限网络若失败见末尾 troubleshooting
%pip install -q -U unsloth

## 2. 环境自检
检查 GPU、显存、bf16 支持（T4 应为 False，印证为何用 fp16）与各库版本。
注意：只查版本**不 import unsloth**——unsloth 一旦被 import 会 monkey-patch `trl.SFTTrainer`，污染后面原生的 HF 路线。

In [ ]:
import platform, subprocess, torch
from importlib.metadata import version, PackageNotFoundError
print("python", platform.python_version(), "| torch", torch.__version__,
      "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported(), " <- T4 应为 False => 用 fp16")
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"VRAM total {total_b/1e9:.1f} GB | free {free_b/1e9:.1f} GB")
    try:
        print(subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"]).decode().strip())
    except Exception as e:
        print("nvidia-smi n/a:", e)
else:
    print("!! 未检测到 GPU，本 notebook 需要 GPU 环境")
# 只查版本、不 import ——尤其 unsloth：import 会 monkey-patch trl.SFTTrainer，污染后面的 HF 路线
for p in ["transformers", "trl", "peft", "datasets", "accelerate", "unsloth"]:
    try:
        print(p, version(p))
    except PackageNotFoundError:
        print(p, "未安装")

## 3. 共用配置
三个方案读同一份 `CFG`，保证 LoRA 秩、epoch、学习率、batch 完全一致，对比才公平。数据三分为 **train / val / test**：val 用于训练中算验证损失看过拟合，test 独立留给最终生成评测。想更容易看到过拟合就把 `n_train` 调小或 `num_epochs` 调大。

In [ ]:
# 三个方案共用的配置（内存变量，保证对比公平）
CFG = {
    "model_id": "Qwen/Qwen3-0.6B",
    "max_seq_length": 512,
    "lora_r": 16, "lora_alpha": 16,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    # 数据三分：train 训练 / val 训练中算验证损失(看过拟合) / test 最终生成评测
    "n_train": 400, "n_val": 100, "n_test": 60,
    "batch_size": 4, "grad_accum": 2,
    "num_epochs": 6,          # 用 epoch 而非固定步数，训得久一点才看得到过拟合
    "eval_steps": 20,         # 每 20 步在 val 上评一次；同时作为 logging_steps
    "lr": 2e-4, "warmup": 15, "seed": 3407,
}
# 想更容易看到过拟合：把 n_train 调小(如 150)或 num_epochs 调大(如 12)
CFG

## 4. 生成数据集（base/HF/unsloth 共用）
合成「临床短句→JSON」抽取任务。**flag 直接写在句子里 → 这是纯格式/行为任务，不是知识注入**（呼应 F1：知识用 RAG，行为用微调）。**去重后切分 train/val/test，三份无重叠（无数据泄漏）**。真实场景请替换为脱敏后的自有数据。

In [ ]:
# 生成合成"临床短句 -> 结构化 JSON 抽取"数据集（全在内存，供 base/HF/unsloth 共用）
# flag 字段直接写在句子里 => 纯格式转换任务(行为，非知识)，呼应 F1 头号命题
import json, random
random.seed(0)

SEX = ["male", "female"]
# (检验项, 单位, 取值范围, 正常范围)
TESTS = [
    ("hemoglobin", "g/dL", (6, 18), (12, 16)),
    ("glucose", "mg/dL", (50, 400), (70, 110)),
    ("creatinine", "mg/dL", (0.3, 8.0), (0.6, 1.3)),
    ("potassium", "mmol/L", (2.0, 7.0), (3.5, 5.1)),
    ("WBC", "10^9/L", (1.0, 30.0), (4.0, 11.0)),
]
INSTR = ("Extract the fields (age, sex, test, value, unit, flag) from the clinical "
         "note and return ONLY a JSON object.")

def make():
    age = random.randint(18, 89)
    sex = random.choice(SEX)
    name, unit, (lo, hi), (nlo, nhi) = random.choice(TESTS)
    val = round(random.uniform(lo, hi), 1)
    flag = "normal" if nlo <= val <= nhi else ("high" if val > nhi else "low")
    text = random.choice([
        f"A {age}-year-old {sex} patient had {name} measured at {val} {unit}, flagged as {flag}.",
        f"Lab report - {sex}, {age} yo: {name} = {val} {unit} ({flag}).",
        f"{name} for a {sex} aged {age} came back {val} {unit}, considered {flag}.",
    ])
    out = {"age": age, "sex": "M" if sex == "male" else "F", "test": name,
           "value": val, "unit": unit, "flag": flag}
    return {"input": INSTR + "\n\nNote: " + text,
            "output": json.dumps(out, ensure_ascii=False)}

# 去重后再切分 —— 保证 train/val/test 之间无重复样本（无数据泄漏，这本身是判质点）
need = CFG["n_train"] + CFG["n_val"] + CFG["n_test"]
seen, records = set(), []
while len(records) < need:
    r = make()
    if r["input"] in seen:
        continue
    seen.add(r["input"]); records.append(r)

nt, nv = CFG["n_train"], CFG["n_val"]
train_rows = records[:nt]
val_rows = records[nt:nt + nv]
test_rows = records[nt + nv:]
print(f"train {len(train_rows)} | val {len(val_rows)} | test {len(test_rows)}  （已去重，三份无重叠）")
print("--- 样例 ---")
print(train_rows[0]["input"])
print("=>", train_rows[0]["output"])

## 5. 共用工具函数
生成、评估（合法 JSON 率 / 精确匹配）、显存清理、TRL 版本容错都封装在这里，三个方案复用，口径一致。运行本 cell 后 `results = {}` 备好。

In [ ]:
# 共用工具函数（生成 / 评估 / 显存清理）——各阶段复用，保证口径一致
import gc, time, json, torch
from datasets import Dataset

def to_messages(r):
    return [{"role": "user", "content": r["input"]},
            {"role": "assistant", "content": r["output"]}]

def format_text(tok, r):
    msgs = to_messages(r)
    try:
        return tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False)

def build_train_dataset(tok):
    # 预先套 chat 模板成纯文本字段 "text"：vanilla TRL 与 unsloth-patched TRL 都能直接吃，
    # 不需要 formatting_func，绕开各版本对 messages 对话格式处理差异导致的报错。
    return Dataset.from_list([{"text": format_text(tok, r)} for r in train_rows])

def build_val_dataset(tok):
    return Dataset.from_list([{"text": format_text(tok, r)} for r in val_rows])

def parse(s):
    """从模型输出里抠出第一个 JSON 对象并解析；失败返回 None。"""
    try:
        i, j = s.find("{"), s.rfind("}")
        return json.loads(s[i:j + 1]) if i >= 0 and j > i else None
    except Exception:
        return None

def generate(model, tok, msgs):
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**ids, max_new_tokens=128, do_sample=False,
                           pad_token_id=tok.pad_token_id)
    return tok.decode(g[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, tok, build_msgs):
    """在 test 集上生成，算合法 JSON 率 + 精确匹配。"""
    gens = []
    for r in test_rows:
        gens.append({"input": r["input"], "gold": r["output"],
                     "pred": generate(model, tok, build_msgs(r))})
    valid = sum(1 for x in gens if parse(x["pred"]) is not None)
    exact = sum(1 for x in gens if parse(x["pred"]) is not None
                and parse(x["pred"]) == parse(x["gold"]))
    return {"valid_json_rate": round(valid / len(gens), 3),
            "exact_match": round(exact / len(gens), 3), "samples": gens}

def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def sft_config(output_dir):
    from trl import SFTConfig
    common = dict(
        output_dir=output_dir,
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        per_device_eval_batch_size=8,
        num_train_epochs=CFG["num_epochs"], learning_rate=CFG["lr"],
        warmup_steps=CFG["warmup"], logging_steps=CFG["eval_steps"],
        fp16=True, bf16=False, optim="adamw_torch",      # T4: fp16 不用 bf16
        lr_scheduler_type="linear", seed=CFG["seed"], report_to="none", packing=False)
    # eval 策略参数名跨版本有 eval_strategy / evaluation_strategy 之分；文本字段/关 completion-only 也逐级退化
    for strat in (dict(eval_strategy="steps", eval_steps=CFG["eval_steps"]),
                  dict(evaluation_strategy="steps", eval_steps=CFG["eval_steps"]),
                  dict()):
        for extra in (dict(completion_only_loss=False, dataset_text_field="text"),
                      dict(dataset_text_field="text"), dict()):
            try:
                return SFTConfig(**common, **strat, **extra)
            except TypeError:
                continue
    return SFTConfig(**common)

def make_trainer(model, tok, args, **kw):
    """喂预格式化好的 train/val text 数据集；兼容新旧 TRL 的 processing_class / tokenizer 参数名。"""
    from trl import SFTTrainer
    tr, va = build_train_dataset(tok), build_val_dataset(tok)
    try:
        return SFTTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                          processing_class=tok, **kw)
    except TypeError:
        return SFTTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                          tokenizer=tok, **kw)

def last_loss(trainer):
    return next((r["loss"] for r in reversed(trainer.state.log_history) if "loss" in r), None)

def show_curve(trainer, tag):
    """打印 step | train | val 损失曲线，并给出过拟合判定。"""
    tr, ev = {}, {}
    for r in trainer.state.log_history:
        s = r.get("step")
        if "loss" in r: tr[s] = r["loss"]
        if "eval_loss" in r: ev[s] = r["eval_loss"]
    print(f"[{tag}] step | train | val")
    for s in sorted(set(tr) | set(ev)):
        t = f"{tr[s]:.3f}" if s in tr else "  -  "
        v = f"{ev[s]:.3f}" if s in ev else "  -  "
        print(f"      {s:>4} | {t:>5} | {v:>5}")
    if ev:
        bs = min(ev, key=ev.get); lastv = ev[max(ev)]
        if lastv > ev[bs] * 1.05:
            print(f"      ⚠ 验证损失在第 {bs} 步最低({ev[bs]:.3f})后回升到 {lastv:.3f} —— 过拟合迹象，"
                  f"该在 ~{bs} 步早停")
        else:
            print(f"      验证损失最低在第 {bs} 步({ev[bs]:.3f})，末尾未明显回升 —— 尚未过拟合")
    return {"train": tr, "eval": ev}

results = {}   # 汇总各方案结果

## 6. 方案①：不微调的 base 模型（对照）
先量出「未微调」水平（零样本 + 少样本）。看点：base 大概率能吐出结构大致对的 JSON，但会栽在**隐含约定**（`sex` 未归一成 M/F）和**输出干净度**上 → 精确匹配低。少样本能部分补救，但每次推理都要塞示例。

In [ ]:
# 方案①：不微调的 base 模型（零样本 + 少样本），作为微调价值的对照。在训练之前跑。
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_base():
    tok = AutoTokenizer.from_pretrained(CFG["model_id"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # T4：fp16 + sdpa（禁 bf16 / FlashAttention-2）
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_id"], torch_dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()

    zeroshot = evaluate(model, tok, lambda r: [{"role": "user", "content": r["input"]}])

    shots = train_rows[:3]
    def fewshot_msgs(r):
        m = []
        for ex in shots:
            m += [{"role": "user", "content": ex["input"]},
                  {"role": "assistant", "content": ex["output"]}]
        m.append({"role": "user", "content": r["input"]})
        return m
    fewshot = evaluate(model, tok, fewshot_msgs)

    del model
    return {"zeroshot": zeroshot, "fewshot": fewshot}

results["base"] = run_base(); free()
b = results["base"]
print("base 零样本 : validJSON %.0f%% | exact %.0f%%" % (
    b["zeroshot"]["valid_json_rate"] * 100, b["zeroshot"]["exact_match"] * 100))
print("base 少样本 : validJSON %.0f%% | exact %.0f%%" % (
    b["fewshot"]["valid_json_rate"] * 100, b["fewshot"]["exact_match"] * 100))

## 7. 方案②：HuggingFace 微调（transformers + trl + peft）
**模型按 fp32 加载**，fp16 混合精度由 `SFTConfig(fp16=True)` 负责（LoRA 参数梯度须为 fp32，直接 fp16 加载会报 `unscale FP16 gradients`）。`SFTTrainer` + `peft` LoRA，全序列 SFT。
⚠️ 若这里 traceback 出现 `UnslothSFTTrainer`，说明内核里已 import 过 unsloth（patch 了 trl）——**Restart Kernel 从头跑**，保证 HF 用原生 TRL。
训练中每 `eval_steps` 步在 val 上算一次验证损失，末尾打印 `step | train | val` 曲线：train 一路降、val 若在某步后回升即**过拟合**（该早停）。

In [ ]:
# 方案②：HuggingFace 路线 —— transformers + trl(SFTTrainer) + peft(LoRA)
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig

def run_hf():
    tok = AutoTokenizer.from_pretrained(CFG["model_id"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # 训练时按 fp32 加载：fp16=True 的 AMP 需要可训练(LoRA)参数梯度为 fp32，
    # 若直接 fp16 加载会报 "Attempting to unscale FP16 gradients"。fp16 混合精度由 SFTConfig(fp16=True) 负责。
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_id"], attn_implementation="sdpa").to("cuda")
    model.config.use_cache = False

    lora = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.0,
                      target_modules=CFG["target_modules"], bias="none", task_type="CAUSAL_LM")

    # 统一用全序列 SFT（不启用回答段掩码，避免依赖模板 {% generation %} 特性；见 F5 判质#2）
    print("[hf] loss = 全序列 SFT")
    trainer = make_trainer(model, tok, sft_config("out_hf"), peft_config=lora)

    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    out = trainer.train(); train_time = time.time() - t0
    peak = torch.cuda.max_memory_reserved() / 1e9
    show_curve(trainer, "hf")   # train vs val 损失曲线 + 过拟合判定

    m = trainer.model; m.config.use_cache = True; m.eval()
    ev = evaluate(m, tok, lambda r: [{"role": "user", "content": r["input"]}])
    res = {"framework": "HF", "train_time_s": round(train_time, 1),
           "peak_vram_gb": round(peak, 2),
           "samples_per_s": round(out.metrics.get("train_samples_per_second", 0.0), 2),
           "final_loss": last_loss(trainer), **ev}
    del trainer, model, m
    return res

results["hf"] = run_hf(); free()
print({k: results["hf"][k] for k in
       ["train_time_s", "peak_vram_gb", "final_loss", "valid_json_rate", "exact_match"]})

## 8. 方案③：unsloth 微调（FastLanguageModel）
内核优化，主打更快更省显存。`load_in_4bit=False` 用 16-bit LoRA 与 HF 对齐。
⚠️ **单 kernel 说明**：前面已 import 过 transformers，这里 import unsloth 会提示「应在 transformers 之前 import」——功能正常，但显存/速度数字会有小出入。想要最干净的 unsloth 测量：**Restart Kernel**，只跑 1/2/3/4/5 + 本节。import 失败会打印并优雅跳过（只出 base + HF 结果）。

In [ ]:
# 方案③：unsloth 路线 —— FastLanguageModel + trl(SFTTrainer)
# 注意：单 kernel 里前面已 import 过 transformers，此处 import unsloth 会提示
# "应在 transformers 之前 import"。功能正常，仅显存/速度数字略有出入（见本节说明）。
def run_unsloth():
    try:
        from unsloth import FastLanguageModel
    except Exception as e:
        print("[unsloth] import 失败，跳过本路线：", repr(e))
        return None

    model, tok = FastLanguageModel.from_pretrained(
        model_name=CFG["model_id"], max_seq_length=CFG["max_seq_length"],
        dtype=torch.float16, load_in_4bit=False)          # 16-bit LoRA，与 HF 对齐
    model = FastLanguageModel.get_peft_model(
        model, r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.0,
        target_modules=CFG["target_modules"], bias="none",
        use_gradient_checkpointing=False, random_state=CFG["seed"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print("[unsloth] loss = 全序列 SFT")
    trainer = make_trainer(model, tok, sft_config("out_unsloth"))

    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    out = trainer.train(); train_time = time.time() - t0
    peak = torch.cuda.max_memory_reserved() / 1e9
    show_curve(trainer, "unsloth")

    FastLanguageModel.for_inference(model)
    ev = evaluate(model, tok, lambda r: [{"role": "user", "content": r["input"]}])
    res = {"framework": "unsloth", "train_time_s": round(train_time, 1),
           "peak_vram_gb": round(peak, 2),
           "samples_per_s": round(out.metrics.get("train_samples_per_second", 0.0), 2),
           "final_loss": last_loss(trainer), **ev}
    del trainer, model
    return res

results["unsloth"] = run_unsloth(); free()
if results["unsloth"]:
    print({k: results["unsloth"][k] for k in
           ["train_time_s", "peak_vram_gb", "final_loss", "valid_json_rate", "exact_match"]})

## 9. 对比结果
**怎么读这张表**：
- **微调价值 = 能力阶梯**：base 零样本(不行) → few-shot(够用则先用) → 微调。简单任务上few-shot 往往就能满分——这不奇怪，正是 F1 的能力阶梯。微调的价值在于**零样本就达到 few-shot 质量**，推理时不用每次塞示例 → **降本降延迟**。（任务复杂到 few-shot 覆盖不全时，微调还会有质量优势，那是另一个场景。）
- **框架对比**：unsloth 通常更省显存、略快；0.6B 上速度差距不明显，放大到 7B 更显著。
- **公平性提示**：本次 HF 按 **fp32** 加载（保证 fp16 训练正确性）、unsloth 按 **fp16** 加载，所以显存差**有一部分来自 dtype 加载策略，不是纯 kernel 优化**——这正是第 10 节要练的判质。

In [ ]:
# 汇总对比：不微调(base) vs 微调(HF / unsloth)
b, hf, us = results.get("base"), results.get("hf"), results.get("unsloth")

rows = []  # 方案, 训练秒, 峰值显存, 末loss, 合法JSON率, 精确匹配
if b:
    rows.append(("base 零样本", "-", "-", "-",
                 b["zeroshot"]["valid_json_rate"], b["zeroshot"]["exact_match"]))
    rows.append(("base 少样本(3例)", "-", "-", "-",
                 b["fewshot"]["valid_json_rate"], b["fewshot"]["exact_match"]))
if hf:
    rows.append(("HF 微调", hf["train_time_s"], hf["peak_vram_gb"], hf["final_loss"],
                 hf["valid_json_rate"], hf["exact_match"]))
if us:
    rows.append(("unsloth 微调", us["train_time_s"], us["peak_vram_gb"], us["final_loss"],
                 us["valid_json_rate"], us["exact_match"]))

hdr = ["方案", "训练秒", "峰值显存GB", "末loss", "合法JSON率", "精确匹配"]
tbl = [hdr] + [[str(x) for x in r] for r in rows]
w = [max(len(tbl[i][c]) for i in range(len(tbl))) for c in range(len(hdr))]
for i, row in enumerate(tbl):
    print(" | ".join(row[c].ljust(w[c]) for c in range(len(hdr))))
    if i == 0:
        print("-+-".join("-" * w[c] for c in range(len(hdr))))

print()
if b and hf:
    z, fs, y = (b["zeroshot"]["exact_match"], b["fewshot"]["exact_match"], hf["exact_match"])
    print(f"★ 能力阶梯（精确匹配）：base 零样本 {z*100:.0f}% -> few-shot {fs*100:.0f}% -> 微调 {y*100:.0f}%")
    print("  · 不微调(零样本)不行；few-shot 够用则先用它（F1 能力阶梯）")
    print("  · 微调 = 零样本就达到 few-shot 质量 -> 推理不用每次塞示例 -> 降本降延迟（这才是简单任务上微调的价值）")
if hf and us and us.get("train_time_s"):
    print(f"★ 框架对比：unsloth 训练快 {hf['train_time_s']/us['train_time_s']:.2f}x | "
          f"峰值显存 {us['peak_vram_gb']/hf['peak_vram_gb']:.2f}x (越小越省)")
    print("  · 注意：本次 HF 按 fp32 加载、unsloth 按 fp16，显存差有一部分来自 dtype 策略，非纯 kernel 优化")

### 并排看生成（格式遵循质量）

In [ ]:
# 并排看生成：base 零样本(未微调) vs 微调后 —— 直观看差在哪
b, hf = results.get("base"), results.get("hf")
bs = b["zeroshot"]["samples"] if b else []
hs = hf["samples"] if hf else []
ref = hs if hs else bs
for i in range(max(len(bs), len(hs))):
    print("-" * 72)
    if i < len(ref):
        print("输入 :", ref[i]["input"].split("Note:")[-1].strip())
        print("gold :", ref[i]["gold"])
    if i < len(bs):
        print("base零样本 :", bs[i]["pred"])
    if i < len(hs):
        print("微调后     :", hs[i]["pred"])

## 10. 判质与观察引导（无标准答案，能说出理由即达标）

对着上面的数字与生成结果，回答：
1. **微调价值**：base 零样本的精确匹配为什么低？看并排生成——错在 `sex` 没归一成 M/F？多了解释文字？微调后是不是这些都干净了？
2. **少样本 vs 微调**：少样本(3例)把精确匹配拉上来多少？代价是什么（每次推理都要带示例 → 更长 prompt、更慢、更贵）？这正是"微调能降本降延迟"的由来。
3. **过拟合**：看 `step | train | val` 曲线——train 一直降时，val 是继续降还是某步后回升？回升点就是该早停的位置。这个简单任务若 val 没回升，说明还没过拟合（把 `n_train` 调小或 `num_epochs` 调大就能复现过拟合）。
4. **速度/显存**：unsloth 快多少、省多少？在 0.6B 上差距明显吗？想想放大到 7B 会怎样。
5. **对比公平性自查**：单 kernel 顺序跑，unsloth 数字可信吗？（transformers 已被 import、显存有无残留、HF 按 fp32 而 unsloth 按 fp16 加载、warmup、下载权重是否同一份……）要更严谨该怎么测？
6. **上手成本**：读两段训练函数，哪套代码量少、心智负担低？unsloth 省的那几行，代价是什么（灵活性 / 版本绑定）？

> **易踩坑**：只看"精确匹配高 / 合法 JSON 率高"就宣布成功，是本课判质#4 要打的靶子——
> 还要测**通用能力有没有退化（灾难性遗忘）**。本对比只测了目标任务，完整评估在 F5 核心 lab 展开。

## 11. Troubleshooting（T4 常见问题）

- **kernel 无 torch（`ModuleNotFoundError: torch`）**：换 GPU PyTorch 内核（`conda_pytorch_p310`），或运行安装 cell 里的 `%pip install torch`。
- **unsloth import/安装失败**：多为依赖版本或网络。可只跑 base + HF（unsloth 那节会打印错误并跳过）。受限网络下 `pip install unsloth` 可能拉不到 `unsloth_zoo`；确认出网或用离线 wheel。
- **CUDA OOM**：调小 `CFG` 的 `batch_size`（4→2/1）或 `max_seq_length`（512→256）；或把 `grad_accum` 调大保持有效 batch。跑完一个方案没释放时，重跑该 cell 前先执行一次 `free()`。
- **bf16 报错 / 训练 NaN**：确认用 **fp16 不是 bf16**（T4 不支持 bf16）。fp16 若偶发 NaN，降 `lr`（2e-4→1e-4）。
- **`SFTConfig` / `SFTTrainer` 报未知参数**：TRL 版本差异，`make_trainer` 已对 `processing_class/tokenizer` 容错；如仍报错 `pip install -U trl`。
- **`torch_dtype` is deprecated 警告**：仅警告不影响运行，保留是为兼容旧版 transformers，忽略即可。
- **下载慢/失败**：设 `HF_ENDPOINT=https://hf-mirror.com` 或提前 `huggingface-cli download Qwen/Qwen3-0.6B`。